In [1]:
import pandas as pd

# Chargement
file_path = "/home/hado/Projets/Projet_vinci_adt/Data/raw/Injury Severity.xlsx"
df = pd.read_excel(file_path)

# 1. Filtre strict sur le secteur de la construction
df_btp = df[df['construction'] == 1.0].copy()

# 2. Suppression des colonnes générant du Data Leakage ou devenues inutiles
cols_to_drop = ['construction', 'fatality']
df_btp = df_btp.drop(columns=cols_to_drop)

# 3. Vérification de la distribution de la cible (degree_of_inj_x) après filtrage
print("--- Distribution de la cible (degree_of_inj_x) sur le jeu BTP ---")
print(df_btp['degree_of_inj_x'].value_counts(dropna=False))

# 4. Vérification du taux de valeurs manquantes sur le jeu BTP
missing_btp = df_btp.isnull().mean() * 100
print("\n--- % Valeurs manquantes restantes ---")
print(missing_btp[missing_btp > 0].sort_values(ascending=False))

--- Distribution de la cible (degree_of_inj_x) sur le jeu BTP ---
degree_of_inj_x
2    5197
1    4351
3     751
Name: count, dtype: int64

--- % Valeurs manquantes restantes ---
sic_code         87.251189
visibility       25.769492
occ_code          7.845422
event_keyword     0.067968
city_x            0.019419
dtype: float64


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# 1. Préparation du dataframe final : Suppression de sic_code
df_btp = df_btp.drop(columns=['sic_code'])

# 2. Séparation des variables explicatives (X) et de la cible (y)
# Nous mettons de côté les colonnes de texte libre ('abstract', 'event_keyword', 'description') pour le pipeline NLP.
cols_to_exclude = ['degree_of_inj_x', 'abstract', 'event_keyword', 'description', 'date']
X_tabular = df_btp.drop(columns=cols_to_exclude)
X_text = df_btp['abstract'] # Nous utiliserons cette colonne pour le NLP
y = df_btp['degree_of_inj_x']

# 3. Séparation Train / Test (80/20) avec stratification
# La stratification garantit que les proportions des classes 1, 2 et 3 sont identiques dans le Train et le Test.
X_tab_train, X_tab_test, X_text_train, X_text_test, y_train, y_test = train_test_split(
    X_tabular, X_text, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Définition des groupes de variables tabulaires
numerical_features = X_tabular.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_features = X_tabular.select_dtypes(include=['object', 'category', 'str']).columns.tolist()

# 5. Création des pipelines de prétraitement tabulaire
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor_tabular = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# 6. Application du prétraitement sur les données d'entraînement
X_tab_train_processed = preprocessor_tabular.fit_transform(X_tab_train)
X_tab_test_processed = preprocessor_tabular.transform(X_tab_test)

print(f"Dimensions de X_tab_train_processed : {X_tab_train_processed.shape}")
print(f"Dimensions de X_tab_test_processed : {X_tab_test_processed.shape}")

Dimensions de X_tab_train_processed : (8239, 3045)
Dimensions de X_tab_test_processed : (2060, 3045)


In [3]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# 1. Vérification de l'accélération matérielle
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Appareil utilisé pour l'inférence NLP : {device}")

# 2. Chargement du modèle d'embedding
model_nlp = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# 3. Encodage des descriptions (Train et Test)
# batch_size=128 permet de saturer les cœurs du GPU pour réduire le temps de calcul
print("Encodage en cours pour X_text_train...")
X_text_train_emb = model_nlp.encode(X_text_train.dropna().tolist(), batch_size=128, show_progress_bar=True)

print("Encodage en cours pour X_text_test...")
X_text_test_emb = model_nlp.encode(X_text_test.dropna().tolist(), batch_size=128, show_progress_bar=True)

# 4. Concaténation des matrices : Tabulaire + NLP
X_train_final = np.hstack((X_tab_train_processed, X_text_train_emb))
X_test_final = np.hstack((X_tab_test_processed, X_text_test_emb))

print(f"\nDimensions du jeu d'entraînement final (X_train_final) : {X_train_final.shape}")
print(f"Dimensions du jeu de test final (X_test_final) : {X_test_final.shape}")

Appareil utilisé pour l'inférence NLP : cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Encodage en cours pour X_text_train...


Batches:   0%|          | 0/65 [00:00<?, ?it/s]

Encodage en cours pour X_text_test...


Batches:   0%|          | 0/17 [00:00<?, ?it/s]


Dimensions du jeu d'entraînement final (X_train_final) : (8239, 3429)
Dimensions du jeu de test final (X_test_final) : (2060, 3429)
